# Chef Agent

Use this notebook to prototype the main Personal Chef Agent workflow.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

print(sys.path[0])

c:\Users\praya\Agentic_PersonalChef


In [2]:
import langchain
import groq
import dotenv
import sounddevice
import scipy

print("All requirements installed successfully")

All requirements installed successfully


In [3]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv

print("Environment working")

Environment working


In [6]:

from config import llm
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client=TavilyClient()

@tool
def web_search(query:str)->Dict[str,Any]:
    """Search the web for information"""

    return tavily_client.search(query)

In [7]:
system_prompt="""You are a personal agentic chef. Your task is to create a personalised recipe 
with the leftover food items provided by the user. Use the web search to provide the recipe to the user.
If the user asks for recipe instruction provide them with the same."""

In [8]:
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent=create_agent(
    model=llm,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [13]:
from langchain.messages import HumanMessage

config={"configurbale":{"thread_id":"1"}}

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke({"messages": [HumanMessage(content="I have some leftover milk and bread left. What can i make?")]},
                        config)

print(response['messages'][-1].content)


You can make a variety of dishes using leftover milk and bread. Here are a few ideas:

1. Milk Toast: A simple dessert made by toasting bread, mixing milk with sugar and vanilla, and then soaking the toast in the milk mixture.
2. Breakfast Casserole: A casserole made by layering toasted or stale bread, eggs, milk, and spices/herbs of choice.
3. Bread Pudding: A creamy pudding made by soaking leftover bread in a mixture of milk, sugar, and eggs, and then baking until set.

Here's a simple recipe for Milk Toast:

Ingredients:

* 2 pieces of bread
* 1/2 cup of milk
* 1 tsp of sugar
* 1/2 tsp of vanilla extract
* Butter for toasting

Instructions:

1. Cut the edges of the bread and spread butter on each piece.
2. Toast the bread until golden brown.
3. Mix milk, sugar, and vanilla extract in a bowl.
4. Soak the toasted bread in the milk mixture until the bread absorbs the liquid.
5. Serve warm and enjoy!

You can find more recipes and detailed instructions by searching online or checking ou

In [33]:
from langgraph.checkpoint.memory import InMemorySaver

from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

from config import vision_llm

image_agent = create_agent(
    model=vision_llm,
    tools=[web_search],
    checkpointer=InMemorySaver()
)

config = {
    "configurable": {
        "thread_id": "inventory-session-1"
    }
}

In [44]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(
    accept="image/*",
    multiple=False
)

display(uploader)

FileUpload(value=(), accept='image/*', description='Upload')

In [65]:
import base64

if not uploader.value:
    raise ValueError("Please upload an image first.")

uploaded_file = uploader.value[0]

image_bytes = uploaded_file["content"]

img_b64 = base64.b64encode(image_bytes).decode("utf-8")

mime_type = uploaded_file["type"]

image_url = f"data:{mime_type};base64,{img_b64}"

In [46]:
from langchain_core.messages import HumanMessage

message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": """
            Analyze this fridge inventory.

            Identify all visible ingredients and food items.
            """
        },
        {
            "type": "image_url",
            "image_url": {
                "url": image_url
            }
        }
    ]
)

In [66]:
from config import vision_llm

vision_response = vision_llm.invoke([message])

inventory_text = vision_response.content

print(inventory_text)

The fridge contains the following visible ingredients and food items:

1. A metal pot with remnants of food
2. Three black plastic containers with lids, likely containing leftovers or prepared meals
3. A white plastic container with a lid
4. A bag of mangoes
5. A loaf of bread
6. Lemons
7. A bag of apples or pears
8. A bag of green apples
9. A bag of oranges
10. A packet of Maggi noodles
11. Possibly some other fruits at the bottom shelf. 

These are the identifiable items in the fridge. The presence of a variety of fruits and a loaf of bread suggests that the fridge is being used to store everyday food items. The metal pot and black plastic containers indicate that the person may have been cooking and storing leftovers.


In [50]:
response = image_agent.invoke(
    {
        "messages": [
            HumanMessage(
                content=f"""
                Here are ingredients available in my fridge:

                {inventory_text}

                Based on these ingredients:

                1. Suggest recipes I can make
                2. Include complete cooking instructions
                3. Suggest healthy alternatives if possible
                4. Give high-protein and quick meal options too

                Format every recipe like this:

                Recipe Name:

                Ingredients:
                - ...

                Instructions:
                1. ...
                2. ...
                3. ...

                Macros:
                1.Protein:
                2.Carbs:
                3.Fats:

                """
            )
        ]
    },
    config=config
)

print(response["messages"][-1].content)

Here are some recipe ideas based on the ingredients you have:

**Recipe 1: Baked Fish with Lemon and Tomatoes**

Ingredients:
- 1 bag of possibly fish
- 2-3 lemons
- 2-3 tomatoes
- Bread (for serving)

Instructions:
1. Preheat oven to 400°F (200°C).
2. Season the fish with salt, pepper, and herbs.
3. Place the fish on a baking sheet lined with parchment paper.
4. Slice the lemons and tomatoes, and place them on top of the fish.
5. Bake for 15-20 minutes or until the fish is cooked through.
6. Serve with bread on the side.

Macros:
1. Protein: 35g (fish) + 2g (lemon) = 37g
2. Carbs: 10g (tomatoes) + 20g (bread) = 30g
3. Fats: 10g (fish) + 0g (lemon) = 10g

**Recipe 2: Fish and Potato Hash**

Ingredients:
- 1 bag of possibly fish
- 1 bag of possibly potatoes or onions
- 2-3 lemons

Instructions:
1. Dice the potatoes and onions.
2. Debone the fish and dice it into small pieces.
3. Fry the potatoes, onions, and fish in butter with your choice of spices.
4. Serve with a squeeze of lemon jui

In [53]:
import os

os.makedirs("recordings", exist_ok=True)

In [72]:
import sounddevice as sd
from scipy.io.wavfile import write

duration = 5
sample_rate = 44100

print("Recording... Speak now.")

audio = sd.rec(
    int(duration * sample_rate),
    samplerate=sample_rate,
    channels=1
)

sd.wait()

audio_path = "recordings/user_audio.wav"

write(audio_path, sample_rate, audio)

print("Audio saved.") 

Recording... Speak now.
Audio saved.


In [73]:
from groq import Groq

from config import GROQ_API_KEY

client = Groq(api_key=GROQ_API_KEY)

with open(audio_path, "rb") as file:
    transcription = client.audio.transcriptions.create(
        file=file,
        model="whisper-large-v3"
    )

user_query = transcription.text

print(user_query)

 Check the inventory if there is chicken.


In [74]:
vision_response = vision_llm.invoke([message])

inventory_text = vision_response.content

print(inventory_text)

The visible ingredients and food items in the fridge are:

* Lemons
* A bag of fruit (possibly apples or pears)
* A bag of potatoes
* A loaf of bread
* A bag of snacks (possibly Maggi)
* A pot of food (possibly leftovers)
* Three containers of food (possibly leftovers or takeout)
* Various fruits (possibly apples, oranges, or other types of fruit) in the bottom drawer.


In [76]:
combined_prompt = f"""
User request:

{user_query}

Detected ingredients:

{inventory_text}

Suggest recipes based on:
- available ingredients
- user request

Include:
- recipe name
- cooking instructions
- preparation time
- healthy alternatives
- quick meals
"""

response = image_agent.invoke(
    {
        "messages": [
            HumanMessage(content=combined_prompt)
        ]
    },
    config=config
)

print(response["messages"][-1].content)

Here are some recipe ideas based on the ingredients you have:

**Recipe 1: Apple Lemon Bread**

Ingredients:
- 1 bag of fruit (possibly apples or pears)
- 1 lemon
- 1 loaf of bread

Instructions:
1. Preheat oven to 350°F (180°C).
2. Grate the apple and mix with lemon juice, sugar, and eggs.
3. Add flour, baking powder, and salt to the mixture.
4. Pour the mixture into a loaf pan and bake for 40-45 minutes.

Preparation Time: 20 minutes
Healthy Alternative: Use whole wheat bread and reduce sugar content.
Quick Meal: Yes, can be made in under an hour.

**Recipe 2: Fruit Salad**

Ingredients:
- 1 bag of fruit (possibly apples or pears)
- Various fruits (possibly apples, oranges, or other types of fruit)

Instructions:
1. Cut the fruits into bite-sized pieces.
2. Mix the fruits together in a bowl.
3. Squeeze a bit of lemon juice on top.

Preparation Time: 10 minutes
Healthy Alternative: Use a variety of fruits to increase nutrient intake.
Quick Meal: Yes, can be made in under 15 minutes.

